# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [2]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [3]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [4]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.co

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [5]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [6]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [7]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [8]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [9]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [10]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [11]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 9 relevant links


{'links': [{'type': 'company page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'social media', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'social media', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'social media',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [ ]:
select_relevant_links("https://huggingface.co")

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [12]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [13]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 13 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
NEW
Storage Buckets: AI-native object storage
GGML and llama.cpp join Hugging Face 🔥
Try HuggingChat Omni – Chat with AI 💬
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Jackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled
Updated
6 days ago
•
58.8k
•
604
fishaudio/s2-pro
Updated
3 days ago
•
3.96k
•
374
Lightricks/LTX-2.3
Updated
9 days ago
•
501k
•
593
HauhauCS/Qwen3.5-9B-Uncensored-HauhauCS-Aggressive
Updated
10 days ago
•
202k
•
413
Qwen/Qwen3.5-9B
Updated
12 days ago
•
1.83M
•
798
Browse 2M+ models
Spaces
Running
on
CPU Upgrade
173
The Synthetic Data Playbook: Generati

In [14]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [15]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [16]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nNEW\nStorage Buckets: AI-native object storage\nGGML and llama.cpp join Hugging Face 🔥\nTry HuggingChat Omni – Chat with AI 💬\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nJackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled\nUpdated\n6 days ago\n•\n58.8k\n•\n604\nfishaudio/s2-pro\nUpdated\n3 days ago\n•\n3.96k\n•\n374\nLightricks/LTX-2.3\nUpdated\n9 days ago\n•\n501k\n•\n593\nHauhauCS/Qwen3.5-9B-Uncensored-HauhauCS-Aggressive\

In [17]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [18]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 8 relevant links


# Hugging Face: Building the Future of AI Together

---

## About Hugging Face

Hugging Face is the thriving collaboration platform for the global machine learning (ML) community. It serves as a central hub where developers, researchers, and AI enthusiasts come together to share, explore, discover, and innovate on open-source machine learning models, datasets, and applications. At the heart of the AI revolution, Hugging Face empowers the next generation of ML engineers, scientists, and end users to collaborate ethically and openly, fostering a future where AI benefits everyone.

---

## What Hugging Face Offers

- **Models**  
  Access and contribute to a vast library of over 2 million open-source machine learning models ranging across various AI tasks including natural language processing, computer vision, audio, and synthetic data.

- **Datasets**  
  Explore and share from over 500,000 public datasets curated and maintained by the community to fuel AI research and applications.

- **Spaces**  
  Host and discover AI applications running on Hugging Face infrastructure, enabling quick experimentation and deployment of innovative ML demos and solutions.

- **Buckets**  
  AI-native object storage designed to seamlessly integrate with machine learning workflows, simplifying data management.

- **HuggingChat Omni**  
  An AI-powered chat interface for interactive communication with AI models directly on the platform.

---

## Community & Collaboration

Hugging Face is more than a platform — it is a dynamic ecosystem with a rapidly growing community at its core. Users collaborate across models, datasets, and applications to accelerate AI advancements. The community-driven approach ensures open, ethical development and democratizes access to cutting-edge technology.

---

## Company Culture

- **Open & Ethical AI:** Committed to building a future with AI that is transparent, fair, and inclusive.  
- **Innovation at the Edge:** Hosting a talented science team pushing boundaries in technology and research.  
- **Community First:** Facilitates collaboration and knowledge-sharing across all skill levels from beginners to experts.  
- **Fast-Paced & Impact-Driven:** Supports rapid development and deployment of AI tools and models that can create real-world impact.

---

## For Customers and Partners

Whether you are an enterprise, startup, or researcher, Hugging Face offers scalable solutions and infrastructure to accelerate your AI initiatives:

- Host unlimited public models, datasets, and applications.
- Build intelligent applications with ease.
- Access enterprise pricing and support for professional needs.

---

## Careers at Hugging Face

Join a passionate team at the forefront of AI innovation. Hugging Face welcomes talent eager to make a difference in machine learning, software engineering, research, and community management.

Openings are continuously updated — visit their Careers page to explore opportunities and contribute to shaping the future of AI.

---

## Connect and Learn More

- Explore the library of models and datasets: [huggingface.co/models](https://huggingface.co/models)  
- Dive into documentation and community forums for support and learning.  
- Follow Hugging Face on GitHub, Twitter, LinkedIn, and Discord to engage with the community.

---

## Brand Identity

- Vibrant colors: Yellow (#FFD21E), Orange (#FF9D00), and Neutral Gray (#6B7280).  
- Friendly and modern brand design reflecting an open, collaborative AI future.

---

**Join Hugging Face today — where the AI community builds the future together.**

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [ ]:
def stream_brochure(company_name, url): #To display in an animated way using the IPython built in function update_display() with the Markdown() function
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [20]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 3 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is a pioneering AI community and collaboration platform dedicated to building the future of machine learning. It serves as a central hub where machine learning engineers, scientists, and enthusiasts can explore, share, and collaborate on open-source models, datasets, and applications.

Hugging Face empowers the global AI community to innovate openly and ethically, driving the next generation of AI technologies with a strong focus on openness, collaboration, and accessibility. With one of the fastest growing communities in AI, Hugging Face is at the heart of the AI revolution, hosting over 2 million models and 500k datasets.

---

## Platform Highlights

- **Models:** Over 2 million open-source machine learning models available for browsing, experimentation, and deployment.
- **Datasets:** Access to 500k+ datasets optimized for ML tasks, supporting synthetic data experiments and real-world applications.
- **Spaces:** Host and discover AI applications running on scalable infrastructure, including CPU and advanced compute options.
- **Storage Buckets:** AI-native object storage designed for seamless handling of data and model storage.
- **HuggingChat Omni:** Interactive chat with AI to explore conversational models and smart assistants.

---

## Customers & Enterprise Solutions

Hugging Face offers dedicated **Team and Enterprise Plans** to scale organizations with the most advanced AI platform featuring:

- **Enterprise-grade security:** Single Sign-On (SSO), audit logs, and granular access control.
- **Flexible storage:** Extra private storage options ensuring data privacy.
- **Advanced compute resources:** Options to boost performance including ZeroGPU quota boosts.
- **Analytics:** Centralized dashboard for tracking and analyzing repository usage.
- **Collaboration:** Private dataset viewers, token management, and resource groups for effective team management.

These offerings help organizations accelerate AI innovation with robust security and support, making Hugging Face the choice platform for tech companies and research institutions worldwide.

---

## Company Culture

At Hugging Face, the core values revolve around:

- **Open collaboration:** Building a welcoming and supportive environment for sharing knowledge and innovations.
- **Ethical AI:** Commitment to creating AI technologies responsibly and transparently.
- **Community-driven development:** Facilitating a space where users and developers contribute, learn, and improve together.
- **Innovation at the edge:** Pioneering new technologies with a talented science team pushing AI capabilities forward.

The company promotes inclusivity and ongoing learning to empower the next generation of AI practitioners.

---

## Careers & Opportunities

Hugging Face is continuously looking for passionate talent who thrive in a fast-paced, cutting-edge AI environment. Career opportunities often include roles in:

- Machine Learning Research and Engineering
- Software Development & Platform Engineering
- Data Science and Analytics
- Community & Developer Relations
- Product Management and Design

Joining Hugging Face means becoming part of a visionary community shaping the future of AI, with opportunities to experiment, collaborate, and learn on a global scale.

---

## Connect & Learn More

- **Explore AI models & datasets:** https://huggingface.co/models  
- **Enterprise Solutions:** https://huggingface.co/enterprise  
- **Careers:** https://huggingface.co/careers  
- **Community & Documentation:** https://huggingface.co/docs  
- **Social:** GitHub, Twitter, LinkedIn, Discord

---

**Hugging Face - The AI community building the future.**

Your gateway to open, ethical, and cutting-edge machine learning innovation.

In [21]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


# Hugging Face Brochure

---

## Who We Are

**Hugging Face** is the AI community building the future of machine learning. We serve as the central collaboration platform for machine learning practitioners worldwide — helping them create, discover, share, and experiment with models, datasets, and AI applications in an open, ethical, and highly collaborative environment.

---

## Our Mission

We empower the next generation of machine learning engineers, data scientists, researchers, and AI end-users to accelerate innovation. Our platform fosters open-source advancement and drives cooperation across the global community to build a transparent and ethical AI future, together.

---

## What We Offer

### The Hub for Machine Learning Collaboration

- **Models:** Access and contribute to over 2 million machine learning models, ranging from natural language processing, computer vision, speech, and more.
- **Datasets:** Browse and share from over 500,000 datasets contributed by the community, enabling robust training and experimentation.
- **Spaces:** Deploy and discover 1 million+ AI applications. Prototype and present models easily through interactive demo spaces running efficiently on platforms like CPU or Zero infrastructure.
- **Storage Buckets:** Our new AI-native object storage built to optimize large-scale model and data hosting.

### Cutting-edge Tools & Community Highlights

- Integration with popular ML tools like GGML and llama.cpp.
- Launch of products like HuggingChat Omni, providing seamless chat interaction with AI.
- Trending projects continuously updated with community contributions, including synthetic data generation, text-to-video applications, image editing via transformers, and more.

---

## Company Culture

Hugging Face values openness, collaboration, and ethical innovation. We cultivate a vibrant, inclusive community fueled by shared knowledge and creativity. We support a fast-moving, open-source ethos that encourages experimentation and learning.

Our open platform invites participation from hobbyists to industry experts, fostering an environment where all ideas are welcomed and rapidly realized.

---

## Our Customers and Community

- **Machine Learning Researchers & Scientists:** To prototype, share, and improve models quickly.
- **Data Scientists:** For access to diverse datasets to train state-of-the-art models.
- **Developers & AI Enthusiasts:** To build and demo new AI applications.
- **Enterprises:** Leveraging scalable, secure ML infrastructure with enterprise-grade solutions for production needs.
- **Educational Institutions and Students:** To learn, experiment, and contribute to open-source AI.

---

## Careers at Hugging Face

Join us if you are passionate about AI and open-source collaboration. Hugging Face looks for talent excited to push the boundaries of machine learning with a commitment to ethical AI development. Our teams thrive in dynamic, innovative environments where individual contributions have large impact.

Opportunities exist across:

- Software Engineering
- Research and Data Science
- Developer Relations and Community Management
- Product Management and Design
- Enterprise Solutions

Check our website for current openings and start contributing to the AI revolution.

---

## Connect with Us

Explore our platform and become part of the machine learning community shaping the future of AI:

- **Website:** https://huggingface.co
- **Explore Models, Datasets, and Spaces:** Browse our extensive collections and contribute  
- **Get Started:** Sign up to host your models or launch your AI applications  
- **Follow Us:** Engage with our active community and stay updated on latest innovations  

---

**Hugging Face — The AI community building the future.**  
Join us and help shape the open, ethical, and collaborative AI ecosystems.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>